# 01 — Data preparation

Builds every dataset artifact the other notebooks consume, from the raw corpora
in `data/`. Every step is **load-default**: an existing artifact is reused as-is
unless the matching `FORCE_REBUILD_*` flag in the config cell is set.

**v1 chain** (legacy SVM / BiLSTM era, kept for comparability)

- `data/processedDataset.csv` — balanced `(text, aiGenerated)` frame from
  `persian_ai_generated_final.csv` + `blogs.csv`
- `data/tokenized_texts.joblib` — shekar lemmatize + tokenize cache
- `data/style_features.joblib`, `data/tfidf_vectorizer_X.joblib` — the ten served
  style features (column order pinned by `aitextd.features.STYLE_FEATURE_NAMES`)
  and the joint TF-IDF + style matrix
- `data/split_indices.joblib` — canonical 80/20 stratified split (seed 42)

**v2 layer** (review-grade evaluation, schema in `aitextd.dataset`)

- `data/dataset_v2.parquet` — metadata-complete, hash- and SimHash-deduplicated
  frame (+ `.csv` and `dataset_v2.manifest.json`)
- `data/splits_v2.joblib`, `data/splits_v2.json` — group-aware indices into the
  parquet for every evaluation condition
- `data/style_features_v2.parquet` — served style features + Persian extensions
- `data/ai_revised.parquet`, `data/paraphrase_attack.parquet` — machine-revision
  class and paraphrase-attack set

Rebuilds are expensive: the human corpus alone is 2.26 GB, tokenization and the
SimHash dedup take minutes to hours, and a full v2 rebuild is only trustworthy
if it passes the parity check at the end of this notebook.

## 0 — Config

`PROJECT_ROOT` comes from `aitextd.paths`. All artifact paths are constructed
here as `PROJECT_ROOT / "data" / ...`: every cache lives under `data/`, and the
stale `aitextd.paths` cache constants (`TOKENIZED_TEXTS_JOBLIB`,
`SPLIT_INDICES_JOBLIB`, ...) point outside `data/` and are deliberately not
used. Cell-local imports (`scipy.sparse`,
`sklearn.model_selection`, ...) sit in the cells that use them.

In [ ]:
import json
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
def _find_project_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "aitextd" / "paths.py").is_file():
            return base
    raise RuntimeError("project root containing aitextd/ not found; run the notebook from the repo")
PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from aitextd import dataset as ds, paths  # noqa: E402
from aitextd.features import PUNCTUATION, STYLE_FEATURE_NAMES, STYLE_FEATURES_VERSION, compute_style_features  # noqa: E402
PROJECT_ROOT = paths.PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "data"  # path convention: everything under PROJECT_ROOT/data
MODELS_DIR = PROJECT_ROOT / "models"
RAW_AI_CSV = DATA_DIR / "persian_ai_generated_final.csv"; RAW_HUMAN_CSV = DATA_DIR / "blogs.csv"
PROCESSED_CSV = DATA_DIR / "processedDataset.csv"; TOKENIZED_JOBLIB = DATA_DIR / "tokenized_texts.joblib"
STYLE_FEATURES_JOBLIB = DATA_DIR / "style_features.joblib"; TFIDF_X_JOBLIB = DATA_DIR / "tfidf_vectorizer_X.joblib"
DEPLOYED_TFIDF_VECTORIZER = MODELS_DIR / "tfidf_vectorizer.joblib"  # used read-only
SPLIT_INDICES_JOBLIB = DATA_DIR / "split_indices.joblib"; HUMAN_DIR = DATA_DIR / "human"
DATASET_V2_PARQUET = DATA_DIR / "dataset_v2.parquet"; DATASET_V2_CSV = DATA_DIR / "dataset_v2.csv"
DATASET_V2_MANIFEST = DATA_DIR / "dataset_v2.manifest.json"
SPLITS_V2_JOBLIB = DATA_DIR / "splits_v2.joblib"; SPLITS_V2_JSON = DATA_DIR / "splits_v2.json"
STYLE_FEATURES_V2_PARQUET = DATA_DIR / "style_features_v2.parquet"
AI_REVISED_PARQUET = DATA_DIR / "ai_revised.parquet"; PARAPHRASE_ATTACK_PARQUET = DATA_DIR / "paraphrase_attack.parquet"
PARAPHRASE_MANIFEST = DATA_DIR / "paraphrase_manifest.json"
SEED = 42
FORCE_REBUILD_V1 = FORCE_REBUILD_V2 = FORCE_REBUILD_ATTACK = False
V2_REBUILT, FROZEN_DOC_IDS, FROZEN_SPLITS = False, None, None  # parity-check plumbing
print("project root:", PROJECT_ROOT, "| data:", DATA_DIR)

## 1 — v1 chain

The original pipeline: balanced two-column dataset, shekar tokenization cache,
style features + joint TF-IDF matrix, one canonical stratified split. Each step
loads its artifact when present (and not forced); the ports below are faithful
to the original builder scripts, with paths re-pointed into `data/`.

### 1a — `data/processedDataset.csv`

AI rows joined with a human subsample matched to the AI count (seed 42),
shuffled. `blogs.csv` is ingested with a plain `pandas.read_csv`, exactly as
the original builder did.

In [ ]:
if PROCESSED_CSV.exists() and not FORCE_REBUILD_V1:
    print(f"processedDataset.csv present ({PROCESSED_CSV.stat().st_size / 1e6:.0f} MB) - load mode")
else:
    ai_df = pd.read_csv(RAW_AI_CSV, encoding="utf-8-sig")
    human_df = pd.read_csv(RAW_HUMAN_CSV, encoding="utf-8-sig")  # 2.26 GB, plain read as in the original
    print(f"AI rows: {len(ai_df)}  human rows: {len(human_df)}")
    human_df = human_df.sample(len(ai_df), random_state=SEED, ignore_index=True)
    ai_part = pd.DataFrame({"text": ai_df["generated_text"], "aiGenerated": 1})
    human_part = pd.DataFrame({"text": human_df["text"], "aiGenerated": 0})
    final = pd.concat([ai_part, human_part]).sample(frac=1, ignore_index=True)
    assert final["text"].isna().sum() == 0, "unexpected NaN texts after the join"
    final.to_csv(PROCESSED_CSV, encoding="utf-8-sig", index=False)
    print(f"wrote {len(final)} rows -> {PROCESSED_CSV.name}")

### 1b — tokenization cache

The shekar pipeline (clean, normalize, lemmatize, word-tokenize) run per
document over the whole v1 corpus, via `aitextd.text`. **A rebuild takes a
long time** (parallelized over all cores, still on the order of an hour) —
hence the cache.

In [ ]:
from aitextd.text import tokenize_persian_text
if TOKENIZED_JOBLIB.exists() and not FORCE_REBUILD_V1:
    tokenized = joblib.load(TOKENIZED_JOBLIB)
    print(f"tokenized cache loaded: {len(tokenized):,} documents")
else:
    import multiprocessing
    df_v1 = pd.read_csv(PROCESSED_CSV)
    print(f"tokenizing {len(df_v1):,} documents on {multiprocessing.cpu_count()} cores (slow)")
    tokenized = joblib.Parallel(n_jobs=multiprocessing.cpu_count())(
        joblib.delayed(tokenize_persian_text)(text) for text in df_v1["text"])
    joblib.dump(tokenized, TOKENIZED_JOBLIB)
    print(f"cached -> {TOKENIZED_JOBLIB.name}")

### 1c — style features + joint TF-IDF matrix

Port of the original style-features builder: the `(n_docs, 10)` style matrix
(cache trusted only when version and shape match), column order pinned by
`STYLE_FEATURE_NAMES` so train/serve cannot drift, then TF-IDF columns
hstacked with the style columns into `tfidf_vectorizer_X.joblib`. The TF-IDF
vectorizer is the **deployed** `models/tfidf_vectorizer.joblib`, loaded
read-only; fitting a *new* vectorizer happens only in a full from-scratch
rebuild when that file does not exist (saved under `data/`, never `models/`).

In [ ]:
from scipy.sparse import csr_matrix, hstack
def build_style_features(texts, cleaned_texts, vectorizer, cache_path=STYLE_FEATURES_JOBLIB):
    # faithful port; cache trusted only when version+shape match; column order
    n, width = len(texts), len(STYLE_FEATURE_NAMES)
    try:
        payload = joblib.load(cache_path)
        matrix = payload["matrix"]
        if payload["version"] == STYLE_FEATURES_VERSION and matrix.shape == (n, width):
            return matrix
    except (FileNotFoundError, KeyError, TypeError, IndexError):
        pass
    rows = []
    for text, cleaned in zip(texts, cleaned_texts, strict=True):
        feats = compute_style_features(str(text), cleaned, vectorizer)
        rows.append([feats[name] for name in STYLE_FEATURE_NAMES])
    matrix = np.asarray(rows, dtype=np.float64)
    joblib.dump({"version": STYLE_FEATURES_VERSION, "matrix": matrix}, cache_path)
    return matrix
if STYLE_FEATURES_JOBLIB.exists() and TFIDF_X_JOBLIB.exists() and not FORCE_REBUILD_V1:
    payload = joblib.load(STYLE_FEATURES_JOBLIB)
    print(f"style matrix loaded: {payload['matrix'].shape} (version {payload['version']})")
    print(f"joint matrix present: {TFIDF_X_JOBLIB.name} ({TFIDF_X_JOBLIB.stat().st_size / 1e6:.0f} MB)")
else:
    df_v1 = pd.read_csv(PROCESSED_CSV)
    tokenized = joblib.load(TOKENIZED_JOBLIB)
    assert len(tokenized) == len(df_v1), "tokenized cache out of date; rebuild cell 1b first"
    cleaned_texts = [" ".join(tokens) for tokens in tokenized]
    if DEPLOYED_TFIDF_VECTORIZER.exists():
        vectorizer = joblib.load(DEPLOYED_TFIDF_VECTORIZER)  # deployed artifact, read-only
    else:
        # from-scratch rebuilds only; models/tfidf_vectorizer.joblib is never written
        vectorizer = TfidfVectorizer(max_features=150_000, min_df=3, ngram_range=(1, 2), sublinear_tf=True)
        vectorizer.fit(cleaned_texts)
        joblib.dump(vectorizer, DATA_DIR / "tfidf_vectorizer_fromscratch.joblib")
        print("no deployed vectorizer found; fitted a new one -> data/tfidf_vectorizer_fromscratch.joblib")
    style = build_style_features(df_v1["text"].astype(str).tolist(), cleaned_texts, vectorizer)
    X = hstack([vectorizer.transform(cleaned_texts), csr_matrix(style)]).tocsr()
    joblib.dump(X, TFIDF_X_JOBLIB)
    print(f"wrote style matrix {style.shape} -> {STYLE_FEATURES_JOBLIB.name}; joint {X.shape} -> {TFIDF_X_JOBLIB.name}")

### 1d — canonical 80/20 split

One stratified split (seed 42) shared by every v1-era model, so they are all
comparable on the same held-out test set. Port of the original split builder.

In [ ]:
from sklearn.model_selection import train_test_split
if SPLIT_INDICES_JOBLIB.exists() and not FORCE_REBUILD_V1:
    v1_split = joblib.load(SPLIT_INDICES_JOBLIB)
else:
    df_v1 = pd.read_csv(PROCESSED_CSV)
    train_idx, test_idx = train_test_split(
        df_v1.index.to_numpy(), test_size=0.2, random_state=SEED, stratify=df_v1["aiGenerated"].to_numpy())
    v1_split = {"train_idx": train_idx, "test_idx": test_idx}
    joblib.dump(v1_split, SPLIT_INDICES_JOBLIB)
print(f"v1 split: train={len(v1_split['train_idx']):,}  test={len(v1_split['test_idx']):,}")

## 2 — v2 layer

The review-grade dataset and evaluation splits, on the canonical schema from
`aitextd.dataset` (labels, domains, origins, generators, revision types,
formality, length buckets — imported, not duplicated). Load-default throughout:
`FORCE_REBUILD_V2` rebuilds `dataset_v2` and everything derived from it
(splits, style features).

### 2a — human domains

`data/human/*.csv`, one CSV per domain, produced by the collector in the next
cell (a compacted port of the original domain collector).

In [ ]:
human_files = sorted(HUMAN_DIR.glob("*.csv")) if HUMAN_DIR.is_dir() else []
for csv_path in human_files:
    print(f"  {csv_path.stem:24s} {len(pd.read_csv(csv_path)):>6,} rows")
if not human_files:
    print("no human domain CSVs under data/human/ - the next cell collects them")

Optional collector — runs **only** when domain files are missing; needs network
and the `datasets` package. Streams the public, license-tagged corpora (ISNA
news apache-2.0; Persian Wikipedia CC-BY-SA-4.0; PersianTwitter ODbL;
iran-legal-persian-qa CC; conversational subtitles CC-BY-4.0, `origin=
translated`; project blog corpus via `processedDataset.csv`), scrubs PII
(emails, URLs, handles, phones, long ids), filters by length window and
Persian-script ratio, and writes one schema-complete CSV per domain plus a
provenance record. Nothing is invented or paraphrased — the raw surface is what
the detectors must face.

In [ ]:
import unicodedata
_PERSIAN = re.compile(r"[\u0600-\u06ff]")
DOMAINS_HUMAN = ("blog", "news", "encyclopedia", "social_media", "legal_admin", "translated_subtitles")
def _scrub_pii(text):
    for pattern, token in ((r"[\w.+-]+@[\w-]+\.[\w.]+", " [EMAIL] "), (r"(https?://\S+|www\.\S+|t\.co/\S+)", " [URL] "),
                           (r"(?<!\w)@[\w\u0600-\u06ff_]+", " [USER] "), (r"(?<!\d)(?:\+?98|0)?9\d{9}(?!\d)", " [PHONE] "),
                           (r"\b\d{10,}\b", " [ID] ")):
        text = re.sub(pattern, token, text)
    return text
def _persian_ratio(text):
    letters = [c for c in text if unicodedata.category(c).startswith("L")]
    return sum(1 for c in letters if _PERSIAN.match(c)) / len(letters) if letters else 0.0
def _keep(text, *, min_chars=200, max_chars=20000, min_persian_ratio=0.85):
    text = text.strip()
    return min_chars <= len(text) <= max_chars and _persian_ratio(text) >= min_persian_ratio
def _strip_wiki_markup(text):
    text = re.sub(r"\{\{[^{}]*\}\}", " ", text)
    text = re.sub(r"<ref[^>]*>.*?</ref>", " ", text, flags=re.DOTALL)
    text = re.sub(r"<[^>]{1,40}>", " ", text)
    text = re.sub(r"\[\[(?:[^|\]]*\|)?([^\]]+)\]\]", r"\1", text)
    return text.replace("'''", "").replace("''", "")
def _clean(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()
def _docs_news(row):
    try:
        meta = json.loads(row.get("METADATA") or "{}")
        extra = json.dumps({"title": meta.get("title", ""), "date": meta.get("jalali-time", "")}, ensure_ascii=False)
    except json.JSONDecodeError:
        extra = "{}"
    return [{"text": _clean(row.get("TEXT")), "author_group": "journalist", "topic_id": "", "extra": extra}]
def _docs_wiki(row):
    title = str(row.get("title") or "")
    if title.startswith(("فهرست", "رده", "ویکی‌پدیا:", "الگو:", "صفحهٔ اصلی")):
        return []
    return [{"text": _clean(_strip_wiki_markup(str(row.get("text") or ""))), "author_group": "wikipedia_editor",
             "topic_id": "", "extra": json.dumps({"title": title, "url": row.get("url", "")}, ensure_ascii=False)}]
def _docs_tweet(row):
    return [{"text": _clean(row.get("Tweets")), "author_group": "tweeter", "topic_id": "",
             "extra": json.dumps({"sentiment": row.get("Label", "")}, ensure_ascii=False)}]
def _docs_legal(row):
    tags = ",".join(row.get("tags") or [])
    qid = json.dumps({"qid": row.get("id", "")}, ensure_ascii=False)
    docs = []
    for role, raw_body in [("legal_asker", row.get("questionBody"))] + [("legal_expert", a) for a in (row.get("answers") or [])[:3]]:
        docs.append({"text": _clean(raw_body), "author_group": role, "topic_id": tags, "extra": qid})
    return docs
COLLECT = {
    "news": ("pourmand1376/isna-news", (), "apache-2.0", "persian", 3000, {"min_chars": 250}, 90, _docs_news),
    "encyclopedia": ("wikimedia/wikipedia", ("20231101.fa",), "cc-by-sa-4.0", "persian", 3000,
                     {"min_chars": 400, "max_chars": 12000}, 25, _docs_wiki),
    "social_media": ("moali-mkh-2000/PersianTwitterDataset-SentimentAnalysis", (), "odbl", "persian", 3000,
                     {"min_chars": 40, "max_chars": 600, "min_persian_ratio": 0.7}, 10, _docs_tweet),
    "legal_admin": ("PerSets/iran-legal-persian-qa", (), "cc", "persian", 3000,
                    {"min_chars": 80, "max_chars": 4000, "min_persian_ratio": 0.8}, 60, _docs_legal),
    "translated_subtitles": ("Maral/conversational-persian-subtitles", (), "cc-by-4.0", "translated", 400,
                             {"min_chars": 400, "max_chars": 20000, "min_persian_ratio": 0.7}, 120, None),
    "blog": ("blogs.csv (project corpus, via processedDataset.csv)", (), "see project README", "persian",
             12000, {"min_chars": 250, "max_chars": 20000, "min_persian_ratio": 0.9}, 2, None),
}
def _domain_rows(domain, limit):
    hf_id, hf_args, _, _, _, keep, cap_factor, docs_fn = COLLECT[domain]
    cap = max(limit * cap_factor, limit + 500)
    if domain == "blog":  # the exact v1 human rows, from processedDataset.csv (no second 2 GB scan)
        kept = 0
        for body in pd.read_csv(PROCESSED_CSV).query("aiGenerated == 0")["text"]:
            body = _clean(body)
            if kept < limit and _keep(body, **keep):
                kept += 1
                yield {"text": body, "author_group": "blogger", "topic_id": "", "extra": "{}"}
        return
    from datasets import load_dataset
    if domain == "translated_subtitles":  # subtitles arrive one line per row; a document is one episode
        episodes, scanned = {}, 0
        for row in load_dataset(hf_id, *hf_args, split="train", streaming=True):
            if scanned >= cap:
                break
            scanned += 1
            key = (str(row.get("movie_name") or ""), str(row.get("episode_id") or ""))
            line = str(row.get("subtitle_text") or "").strip()
            if line:
                episodes.setdefault(key, []).append((str(row.get("start_time") or ""), line))
        kept = 0
        for (movie, episode), lines in sorted(episodes.items()):
            lines.sort(key=lambda item: item[0])
            body = re.sub(r"\s+", " ", " ".join(line for _, line in lines)).strip()
            if kept < limit and _keep(body, **keep):
                kept += 1
                yield {"text": body, "author_group": "subtitle_translator", "topic_id": "",
                       "extra": json.dumps({"movie": movie, "episode": episode}, ensure_ascii=False)}
        return
    kept = scanned = 0
    for row in load_dataset(hf_id, *hf_args, split="train", streaming=True):
        if kept >= limit or scanned >= cap:
            break
        scanned += 1
        for doc in docs_fn(row):
            if _keep(doc["text"], **keep):
                kept += 1
                yield doc
def _collect_domain(domain, limit=None):
    hf_id, _, license_id, origin, default_limit, _, _, _ = COLLECT[domain]
    rows = []
    for row in _domain_rows(domain, limit or default_limit):
        raw_text = row["text"]
        row["text"] = _scrub_pii(raw_text)
        row["pii_scrubbed"] = int(row["text"] != raw_text)
        rows.append(row)
    if not rows:
        raise RuntimeError(f"no rows collected for {domain}")
    out = pd.DataFrame({
        "doc_id": [ds.stable_doc_id(r["text"]) for r in rows], "text": [r["text"] for r in rows],
        "label": ds.HUMAN, "label_binary": 0, "domain": domain, "source": hf_id,
        "license": license_id, "origin": origin, "generator": "human", "provider": "",
        "prompt_id": "", "prompt_style": "", "topic_id": [r["topic_id"] for r in rows],
        "revision_type": "none", "paraphrase_method": "", "author_group": [r["author_group"] for r in rows],
        "non_native_flag": int(origin == "translated"), "pii_scrubbed": [r["pii_scrubbed"] for r in rows],
        "extra": [r["extra"] for r in rows],
    })
    out["length_words"] = out["text"].map(lambda t: len(t.split()))
    out["length_bucket"] = out["length_words"].map(ds.length_bucket)
    out["formality"] = out["text"].map(ds.formality_label)
    HUMAN_DIR.mkdir(parents=True, exist_ok=True)
    out[[c for c in ds.COLUMNS if c in out.columns]].to_csv(
        HUMAN_DIR / f"{domain}.csv", index=False, encoding="utf-8-sig")
    return {"domain": domain, "source": hf_id, "license": license_id, "origin": origin, "rows": int(len(out)),
            "unique_doc_ids": int(out["doc_id"].nunique()), "pii_scrubbed_rows": int(out["pii_scrubbed"].sum()),
            "file": f"{domain}.csv", "collected_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
missing = [d for d in DOMAINS_HUMAN if not (HUMAN_DIR / f"{d}.csv").exists()]
if not missing:
    print("all human domain CSVs present - collection skipped")
else:
    prov_path = HUMAN_DIR / "provenance.json"
    provenance = json.loads(prov_path.read_text(encoding="utf-8")) if prov_path.exists() else {}
    failures = {}
    for domain in missing:
        try:
            provenance[domain] = _collect_domain(domain)
            print(f"[{domain}] collected {provenance[domain]['rows']:,} rows")
        except Exception as exc:  # one flaky source must not block the rest
            failures[domain] = f"{type(exc).__name__}: {exc}"
            print(f"[{domain}] FAILED: {failures[domain]}")
    provenance["_failures"] = failures
    prov_path.write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")

### 2b — `data/dataset_v2.parquet`

Condensed port of the original v2 dataset builder (hash-dedup + SimHash
semantics kept in full). In order: AI corpus with full metadata + every human
domain + machine-revised rows (if present) → exact de-dup by content hash →
near-duplicate removal by banded 64-bit SimHash (Hamming ≤ 3) within each
`(label, domain)` block → cross-label passes (a human row mirroring an AI row
is dropped from the human side; a revised row is dropped against any primary
document *except its own source*) → per-domain TF-IDF MiniBatchKMeans topic
groups for human rows (AI keeps raw topic strings; revisions inherit their
source's group) → derived columns and `extra.v1_split` membership → sort,
validate, write parquet + csv + manifest.

In [ ]:
from collections import Counter
MIN_AI_WORDS, NEAR_DUP_HAMMING, KMEANS_CLUSTERS = 20, 3, 200
def _collapse(text):
    return re.sub(r"\s+", " ", str(text)).strip()
def _load_ai_rows():
    raw = pd.read_csv(RAW_AI_CSV, encoding="utf-8-sig").rename(columns={"generated_text": "text", "model": "generator"})
    raw["text"] = raw["text"].map(_collapse)
    raw = raw[raw["text"].map(lambda t: len(t.split())) >= MIN_AI_WORDS].copy()
    extra_cols = [c for c in ("temperature", "top_p", "char_count", "word_count") if c in raw.columns]
    empty = pd.Series([""] * len(raw))
    get = lambda col: raw.get(col, empty).fillna("").astype(str)
    return pd.DataFrame({
        "doc_id": raw["text"].map(ds.stable_doc_id), "text": raw["text"],
        "generator": raw["generator"].fillna("unknown").astype(str),
        "provider": get("provider"), "prompt_id": get("prompt_style"), "prompt_style": get("prompt_style"),
        "topic_id": get("topic"),
        "extra": raw[extra_cols].apply(lambda row: json.dumps(row.to_dict(), ensure_ascii=False), axis=1),
        **{"label": ds.AI, "label_binary": 1, "domain": "ai_multi", "origin": "persian",
           "source": "persian_ai_generated_final.csv", "license": "apache-2.0 (released dataset)",
           "revision_type": "none", "paraphrase_method": "", "author_group": "llm",
           "non_native_flag": 0, "pii_scrubbed": 0},
    })
def _load_revised_rows():
    if not AI_REVISED_PARQUET.exists():
        return None, {"present": False}
    frame = pd.read_parquet(AI_REVISED_PARQUET)
    frame["text"] = frame["text"].map(_collapse)
    frame = frame[[c for c in ds.COLUMNS if c in frame.columns]]
    prov = {"present": True, "file": AI_REVISED_PARQUET.name, "rows": int(len(frame)),
            "methods": frame["revision_type"].value_counts().to_dict(),
            "rewriters": frame["generator"].value_counts().to_dict(),
            "domains": frame["domain"].value_counts().to_dict()}
    return frame, prov
def _load_human_rows():
    frames, provenance = [], {}
    for csv_path in sorted(HUMAN_DIR.glob("*.csv")):
        frame = pd.read_csv(csv_path, encoding="utf-8-sig")
        frame["text"] = frame["text"].map(_collapse)
        frames.append(frame)
        provenance[csv_path.stem] = {"rows": int(len(frame)), "file": csv_path.name}
    if not frames:
        raise RuntimeError(f"no human domain CSVs under {HUMAN_DIR}; run the collector cell above")
    return pd.concat(frames, ignore_index=True), provenance
def _v1_membership():
    # doc_id -> v1 split, for data the already-trained checkpoints saw
    split = joblib.load(SPLIT_INDICES_JOBLIB)
    processed = pd.read_csv(PROCESSED_CSV)
    mapping = {}
    for name, idx in (("train", split["train_idx"]), ("test", split["test_idx"])):
        for text in processed.iloc[idx]["text"]:
            mapping[ds.stable_doc_id(_collapse(text))] = name
    return mapping
def _revision_sources(frame):
    if "extra" not in frame.columns:
        return [""] * len(frame)
    out = []
    for label, extra in zip(frame["label"], frame["extra"], strict=True):
        source = ""
        if label == ds.AI_REVISED and isinstance(extra, str) and extra:
            try:
                source = json.loads(extra).get("source_doc_id") or ""
            except json.JSONDecodeError:
                source = ""
        out.append(source)
    return out
def _near_dup_mask(texts, sources=None):
    # True where a row near-duplicates an earlier row (banded SimHash); rows
    seen, dup, fingerprints = {}, np.zeros(len(texts), dtype=bool), []
    for i, text in enumerate(texts):
        h = ds.simhash64(text)
        fingerprints.append(h)
        candidates = []
        for b in range(4):
            candidates.extend(seen.get((b, (h >> (16 * b)) & 0xFFFF), []))
        hit = any(not (sources is not None and sources[i] and sources[i] == sources[j])
                  and ds.hamming64(h, fingerprints[j]) <= NEAR_DUP_HAMMING for j in candidates)
        if hit:
            dup[i] = True
            continue
        for b in range(4):
            seen.setdefault((b, (h >> (16 * b)) & 0xFFFF), []).append(i)
    return dup
def _cross_label_drop(frame, primary_labels, secondary_label, skip_source_match):
    # cross-label near-duplicates: the secondary-side row goes, never the primary;
    secondary = frame[frame["label"] == secondary_label]
    primary = frame[frame["label"].isin(primary_labels)]
    hashes = primary["text"].map(ds.simhash64).tolist()
    ids = primary["doc_id"].tolist()
    buckets = {}
    for i, h in enumerate(hashes):
        for b in range(4):
            buckets.setdefault((b, (h >> (16 * b)) & 0xFFFF), []).append(i)
    sources = _revision_sources(frame)
    drops = []
    for idx, row in secondary.iterrows():
        h = ds.simhash64(row["text"])
        hit = False
        for b in range(4):
            for j in buckets.get((b, (h >> (16 * b)) & 0xFFFF), []):
                if skip_source_match and ids[j] == sources[idx]:
                    continue
                if ds.hamming64(h, hashes[j]) <= NEAR_DUP_HAMMING:
                    hit = True
                    break
            if hit:
                break
        if hit:
            drops.append(idx)
    return drops
def _deduplicate(frame):
    stats = {"input_rows": int(len(frame))}
    before = len(frame)
    frame = frame.drop_duplicates(subset="doc_id", keep="first").reset_index(drop=True)
    stats["exact_duplicates_removed"] = int(before - len(frame))
    sources = _revision_sources(frame)
    keep_mask, removed = np.ones(len(frame), dtype=bool), 0
    for (label, domain), group in frame.groupby(["label", "domain"], sort=False):
        positions = group.index.to_numpy()
        group_sources = [sources[i] for i in positions] if label == ds.AI_REVISED else None
        dup_local = _near_dup_mask(group["text"].tolist(), group_sources)
        removed += int(dup_local.sum())
        keep_mask[positions[dup_local]] = False
    frame = frame[keep_mask].reset_index(drop=True)
    stats["near_duplicates_removed"] = removed
    drop = _cross_label_drop(frame, (ds.AI,), ds.HUMAN, False) if (frame["label"] == ds.AI).any() else []
    frame = frame.drop(index=drop).reset_index(drop=True)
    stats["cross_label_near_duplicates_removed"] = len(drop)
    drop = _cross_label_drop(frame, (ds.HUMAN, ds.AI), ds.AI_REVISED, True) if (frame["label"] == ds.AI_REVISED).any() else []
    frame = frame.drop(index=drop).reset_index(drop=True)
    stats["revised_cross_label_duplicates_removed"] = len(drop)
    kept = frame[frame["label"] == ds.AI_REVISED]
    if len(kept):
        per_source = Counter(_revision_sources(kept))
        stats.update({"revised_rows_kept": int(len(kept)), "revised_sources_kept": int(len(per_source)),
                      "revised_kept_per_source_mean": round(len(kept) / max(len(per_source), 1), 2),
                      "revised_methods_kept": kept["revision_type"].value_counts().to_dict()})
    stats["output_rows"] = int(len(frame))
    return frame, stats
def _assign_topic_groups(frame, k, seed):
    # human rows: TF-IDF MiniBatchKMeans clusters within each domain (group ids
    from sklearn.cluster import MiniBatchKMeans
    for domain in sorted(frame.loc[frame["label"] == ds.HUMAN, "domain"].unique()):
        mask = (frame["label"] == ds.HUMAN) & (frame["domain"] == domain)
        texts = frame.loc[mask, "text"]
        if len(texts) < 40:
            frame.loc[mask, "topic_id"] = [f"{domain}_cluster_0"] * len(texts)
            continue
        k_domain = int(min(k, max(2, len(texts) // 20)))
        matrix = TfidfVectorizer(max_features=40000, min_df=3, ngram_range=(1, 1)).fit_transform(texts)
        km = MiniBatchKMeans(n_clusters=k_domain, random_state=seed, n_init=3, batch_size=1024)
        frame.loc[mask, "topic_id"] = [f"{domain}_cluster_{c}" for c in km.fit_predict(matrix)]
    revised_mask = frame["label"] == ds.AI_REVISED
    inherited = 0
    if revised_mask.any():
        doc_to_topic = dict(zip(frame["doc_id"], frame["topic_id"], strict=True))
        for position in frame.index[revised_mask]:
            try:
                source_id = json.loads(frame.at[position, "extra"] or "{}").get("source_doc_id")
            except (TypeError, json.JSONDecodeError):
                source_id = None
            topic = doc_to_topic.get(source_id)
            if topic:
                frame.at[position, "topic_id"] = topic
                inherited += 1
    frame.attrs["revised_topic_inheritance"] = {"revised_rows": int(revised_mask.sum()),
                                                "topic_inherited": inherited,
                                                "source_missing": int(revised_mask.sum()) - inherited}
    return frame
if DATASET_V2_PARQUET.exists() and not FORCE_REBUILD_V2:
    df_v2 = pd.read_parquet(DATASET_V2_PARQUET)
    print(f"dataset_v2 loaded: {len(df_v2):,} rows, {df_v2['label'].value_counts().to_dict()}")
else:
    if DATASET_V2_PARQUET.exists():  # snapshot frozen artifacts for the parity check
        FROZEN_DOC_IDS = pd.read_parquet(DATASET_V2_PARQUET, columns=["doc_id"])["doc_id"].to_numpy()
        FROZEN_SPLITS = joblib.load(SPLITS_V2_JOBLIB) if SPLITS_V2_JOBLIB.exists() else None
    print("building dataset_v2 (SimHash dedup over all rows; several minutes) ...")
    t0 = time.time()
    human, human_prov = _load_human_rows()
    revised, revised_prov = _load_revised_rows()
    frame = pd.concat([_load_ai_rows(), human, *([revised] if revised is not None else [])], ignore_index=True)
    frame, dedup_stats = _deduplicate(frame)
    frame = _assign_topic_groups(frame, KMEANS_CLUSTERS, SEED)
    frame["length_words"] = frame["text"].map(lambda t: len(t.split()))
    frame["length_bucket"] = frame["length_words"].map(ds.length_bucket)
    frame["formality"] = frame["text"].map(ds.formality_label)
    frame["split"] = ""
    v1 = _v1_membership()
    frame["extra"] = [json.dumps({**json.loads(extra or "{}"), **({"v1_split": v1[doc]} if doc in v1 else {})},
                                 ensure_ascii=False)
                      for doc, extra in zip(frame["doc_id"], frame["extra"], strict=True)]
    frame = frame[[c for c in ds.COLUMNS if c in frame.columns]]
    frame = frame.sort_values(["label", "domain", "doc_id"]).reset_index(drop=True)
    ds.validate_frame(frame)
    frame.to_csv(DATASET_V2_CSV, index=False, encoding="utf-8-sig")
    frame.to_parquet(DATASET_V2_PARQUET, index=False)
    generators = frame.loc[frame["label"] != ds.HUMAN, "generator"].value_counts().to_dict()
    manifest = {
        "built_at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "rows": int(len(frame)),
        "label_counts": frame["label"].value_counts().to_dict(),
        "label_domain_counts": {k: {kk: int(vv) for kk, vv in v.items()} for k, v in
                                frame.groupby(["label", "domain"]).size().unstack(fill_value=0).to_dict().items()},
        "origin_counts": frame["origin"].value_counts().to_dict(),
        "formality_counts": frame["formality"].value_counts().to_dict(),
        "length_bucket_counts": frame["length_bucket"].value_counts().to_dict(),
        "n_generators": int(len(generators)), "generators": generators,
        "prompt_style_counts": frame.loc[frame["label"] == ds.AI, "prompt_style"].value_counts().to_dict(),
        "topic_group_counts": {"ai_unique_topics": int(frame.loc[frame["label"] == ds.AI, "topic_id"].nunique()),
                               "human_clusters": int(frame.loc[frame["label"] == ds.HUMAN, "topic_id"].nunique())},
        "v1_membership": dict(Counter(json.loads(e).get("v1_split", "not_in_v1") for e in frame["extra"])),
        "dedup": dedup_stats, "human_provenance": human_prov, "revised_provenance": revised_prov,
        "revised_topic_inheritance": frame.attrs.get("revised_topic_inheritance", {}),
        "files": {DATASET_V2_CSV.name: ds.sha256_file(DATASET_V2_CSV),
                  DATASET_V2_PARQUET.name: ds.sha256_file(DATASET_V2_PARQUET)},
        "dataset_fingerprint": ds.dataset_fingerprint(frame),
        "builder": "notebooks/01_data_preparation.ipynb",
    }
    DATASET_V2_MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    df_v2 = frame
    V2_REBUILT = True
    print(f"dataset_v2 rebuilt in {time.time() - t0:.0f}s: {len(df_v2):,} rows")
    print(f"fingerprint {ds.dataset_fingerprint(frame)[:16]}...  dedup {dedup_stats}")

### 2c — group-aware splits

Compacted port of the original v2 splits builder. Whole groups (AI topics, human
TF-IDF clusters) are assigned largest-first by fill-ratio balancing so no group
straddles splits; machine-revised rows inherit their source document's
partition. Conditions: `in_domain_group` (train/val/test), `cross_generator`,
`cross_domain__<d>`, `cross_style__<s>`, `few_shot__<k>`, `length__short|long`,
`human_vs_ai_revised`, `paraphrase_attention__test`, `v1_test`. Values are
**positional indices into `data/dataset_v2.parquet`**. The original assertion
battery is trimmed to the four leakage-critical checks (disjointness, no group
leak, unseen generators/topics out of train, held-out domain/style out of
train). `split_frames(df, splits, condition)` returns ready-made
`(train, val, test)` frames for one condition.

In [ ]:
TRAIN_FRAC, VAL_FRAC = 0.7, 0.1
FEW_SHOT_KS, ATTACK_SAMPLE = (8, 32, 128), 300
def _grouped_assign(df, mask, seed):
    # largest groups first (random order inside a size class); each goes to the
    positions = np.flatnonzero(mask)
    groups = df.iloc[positions]["topic_id"].astype(str).to_numpy()
    unique_groups = np.unique(groups)
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(unique_groups))
    row_counts = pd.Series(groups).value_counts()
    n_total = len(positions)
    targets = {"train": TRAIN_FRAC * n_total, "val": VAL_FRAC * n_total}
    targets["test"] = n_total - targets["train"] - targets["val"]
    fill = {"train": 0.0, "val": 0.0, "test": 0.0}
    ordered = [unique_groups[p] for p in order]
    ordered.sort(key=lambda g: -int(row_counts[g]))
    group_split = {}
    for group in ordered:
        split = min(fill, key=lambda s: fill[s] / max(targets[s], 1.0))
        group_split[group] = split
        fill[split] += int(row_counts[group])
    return np.array([group_split[g] for g in groups], dtype=object)
def _pick_unseen_generators(df, positions, seed, target_frac=0.2, min_generators=8,
                            min_test_rows=250, test_positions=None):
    # held-out generators, stratified by volume (up to three high-volume, then
    positions = np.asarray(positions, dtype=np.int64)
    generators = df["generator"].astype(str).to_numpy()[positions]
    counts = pd.Series(generators).value_counts()
    counts = counts[counts.index != "unknown"]
    rng = np.random.default_rng(seed)
    target_rows = target_frac * len(positions)
    high = [g for g in counts.index if counts[g] >= 0.01 * len(positions)]
    low = [g for g in counts.index if counts[g] < 0.01 * len(positions)]
    rng.shuffle(high)
    rng.shuffle(low)
    order = high[:3] + low + high[3:]
    unseen, covered = set(), 0
    for generator in order:
        if covered >= target_rows and len(unseen) >= min_generators:
            break
        unseen.add(generator)
        covered += int(counts[generator])
    rounds = 0
    if test_positions is not None:
        in_test = np.zeros(len(df), dtype=bool)
        in_test[np.asarray(test_positions, dtype=np.int64)] = True
        generator_all = df["generator"].astype(str).to_numpy()
        while rounds < len(order):
            usable = np.isin(generator_all, sorted(unseen)) & in_test & (generator_all != "unknown")
            if int(usable.sum()) >= min_test_rows or len(unseen) >= len(counts):
                break
            for generator in order[rounds:]:
                rounds += 1
                if generator not in unseen:
                    unseen.add(generator)
                    covered += int(counts[generator])
                    break
    return unseen
def _balanced(rng, ai_idx, human_idx):
    # cap the larger class down to the smaller, seeded and deterministic
    ai_idx, human_idx = np.asarray(ai_idx, dtype=np.int64), np.asarray(human_idx, dtype=np.int64)
    n = min(len(ai_idx), len(human_idx))
    if len(ai_idx) > n:
        ai_idx = np.sort(rng.choice(ai_idx, size=n, replace=False))
    if len(human_idx) > n:
        human_idx = np.sort(rng.choice(human_idx, size=n, replace=False))
    return ai_idx, human_idx
def build_splits_v2(df, seed=SEED):
    df = df.reset_index(drop=True)
    label, domain = df["label"].to_numpy(), df["domain"].to_numpy()
    is_human, is_ai, is_revised = label == ds.HUMAN, label == ds.AI, label == ds.AI_REVISED
    assignment = np.full(len(df), "", dtype=object)
    for target_label in ds.LABELS:
        for target_domain in sorted(set(domain[label == target_label])):
            mask = (label == target_label) & (domain == target_domain)
            if mask.any():
                assignment[mask] = _grouped_assign(df, mask, seed=seed)
    revised_positions = np.flatnonzero(label == ds.AI_REVISED)
    if revised_positions.size:  # revisions inherit their source document's partition
        docid_to_pos = {doc_id: pos for pos, doc_id in enumerate(df["doc_id"])}
        for pos in revised_positions:
            try:
                source_doc_id = json.loads(df["extra"].iat[pos]).get("source_doc_id")
            except (TypeError, json.JSONDecodeError):
                source_doc_id = None
            source_pos = docid_to_pos.get(source_doc_id)
            if source_pos is not None:
                assignment[pos] = assignment[source_pos]
    df = df.assign(_assignment=assignment)
    def idx(where):
        values = where.to_numpy() if hasattr(where, "to_numpy") else np.asarray(where)
        return np.flatnonzero(values)
    train_idx = idx(df["_assignment"] == "train")
    val_idx = idx(df["_assignment"] == "val")
    test_idx = idx(df["_assignment"] == "test")
    splits = {"in_domain_group__train": train_idx, "in_domain_group__val": val_idx,
              "in_domain_group__test": test_idx}
    # check 1: the primary assignment is pairwise disjoint
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        assert not np.intersect1d(splits[f"in_domain_group__{a}"], splits[f"in_domain_group__{b}"]).size
    # check 2: no whole topic group leaks from train into val/test
    train_groups = set(df.iloc[train_idx]["topic_id"].astype(str))
    for part in ("val", "test"):
        leaked = train_groups & set(df.iloc[splits[f"in_domain_group__{part}"]]["topic_id"].astype(str))
        assert not leaked, f"in_domain_group: {len(leaked)} topic groups leak into {part}"
    ai_test_main = np.intersect1d(test_idx, np.flatnonzero(is_ai))
    unseen = _pick_unseen_generators(df, np.flatnonzero(is_ai), seed, test_positions=ai_test_main)
    unseen_gen_mask = (is_ai & df["generator"].astype(str).isin(sorted(unseen))).to_numpy()
    test_ai_all = np.intersect1d(ai_test_main, np.flatnonzero(unseen_gen_mask))
    seen_ai_mask = (is_ai & ~df["generator"].astype(str).isin(sorted(unseen))).to_numpy()
    val_human = np.intersect1d(val_idx, np.flatnonzero(is_human))
    human_test = np.intersect1d(test_idx, np.flatnonzero(is_human))
    rng = np.random.default_rng(seed)
    ai_side, human_side = _balanced(rng, test_ai_all, human_test)
    splits["cross_generator__train"] = np.sort(np.concatenate([
        np.intersect1d(train_idx, np.flatnonzero(seen_ai_mask)),
        np.intersect1d(train_idx, np.flatnonzero(is_human))]))
    splits["cross_generator__val"] = np.sort(np.concatenate([
        np.intersect1d(val_idx, np.flatnonzero(seen_ai_mask)), val_human]))
    splits["cross_generator__test"] = np.sort(np.concatenate([ai_side, human_side]))
    # check 3: neither an unseen generator nor an unseen test topic appears in training
    assert not set(df.iloc[splits["cross_generator__train"]]["generator"]) & unseen
    assert not set(df.iloc[splits["cross_generator__train"]]["topic_id"]) \
        & set(df["topic_id"].astype(str).iloc[test_ai_all])
    ai_train = np.intersect1d(train_idx, np.flatnonzero(is_ai))
    ai_test_for_domain = np.intersect1d(test_idx, np.flatnonzero(is_ai))
    for held_out in sorted(set(domain[is_human])):
        train_h = np.intersect1d(train_idx, np.flatnonzero(is_human & (domain != held_out)))
        held_out_rows = idx(is_human & (domain == held_out))
        rng = np.random.default_rng(seed)
        ai_side, human_side = _balanced(rng, ai_test_for_domain, held_out_rows)
        name = f"cross_domain__{held_out}"
        splits[f"{name}__train"] = np.sort(np.concatenate([ai_train, train_h]))
        splits[f"{name}__test"] = np.sort(np.concatenate([ai_side, human_side]))
        # check 4a: the held-out domain is fully absent from training
        assert held_out not in set(df.iloc[splits[f"{name}__train"]]["domain"])
    for style in sorted(set(df.loc[is_ai, "prompt_style"])):
        if not style:
            continue
        style_test = np.intersect1d(test_idx, np.flatnonzero(is_ai & (df["prompt_style"] == style).to_numpy()))
        train_ai_other = np.intersect1d(train_idx, np.flatnonzero(is_ai & (df["prompt_style"] != style).to_numpy()))
        rng = np.random.default_rng(seed)
        ai_side, human_side = _balanced(rng, style_test, human_test)
        name = f"cross_style__{style}"
        splits[f"{name}__train"] = np.sort(np.concatenate([
            train_ai_other, np.intersect1d(train_idx, np.flatnonzero(is_human))]))
        splits[f"{name}__test"] = np.sort(np.concatenate([ai_side, human_side]))
        # check 4b: the held-out prompt style is absent from AI training rows
        assert style not in set(df.iloc[splits[f"{name}__train"]].query("label == @ds.AI")["prompt_style"])
    for k in FEW_SHOT_KS:
        rng = np.random.default_rng(seed + k)
        human_pool = np.intersect1d(train_idx, np.flatnonzero(is_human))
        ai_pool = np.intersect1d(train_idx, np.flatnonzero(is_ai))
        assert len(human_pool) >= k and len(ai_pool) >= k, f"few_shot__{k}: not enough training rows"
        splits[f"few_shot__{k}__train"] = np.sort(np.concatenate([
            rng.choice(human_pool, size=k, replace=False), rng.choice(ai_pool, size=k, replace=False)]))
        splits[f"few_shot__{k}__test"] = splits["in_domain_group__test"]
    for bucket in ds.LENGTH_BUCKETS:
        if bucket != "medium":
            splits[f"length__{bucket}__test"] = np.intersect1d(
                test_idx, idx((df["length_bucket"] == bucket).to_numpy()))
    rng = np.random.default_rng(seed + 977)
    attack_pool = np.intersect1d(test_idx, np.flatnonzero(is_ai))
    splits["paraphrase_attention__test"] = (
        np.sort(rng.choice(attack_pool, size=min(ATTACK_SAMPLE, len(attack_pool)), replace=False))
        if len(attack_pool) else np.array([], dtype=np.int64))
    if is_revised.any():
        revised_test = np.intersect1d(test_idx, np.flatnonzero(is_revised))
        revised_train = np.intersect1d(train_idx, np.flatnonzero(is_revised))
        revised_val = np.intersect1d(val_idx, np.flatnonzero(is_revised))
        rng = np.random.default_rng(seed)
        ai_side, human_side = _balanced(rng, revised_test, human_test)
        rng = np.random.default_rng(seed + 3)
        val_side, val_human_side = _balanced(rng, revised_val, np.intersect1d(val_idx, np.flatnonzero(is_human)))
        splits["human_vs_ai_revised__train"] = np.sort(np.concatenate([
            revised_train, np.intersect1d(train_idx, np.flatnonzero(is_human))]))
        splits["human_vs_ai_revised__val"] = np.sort(np.concatenate([val_side, val_human_side]))
        splits["human_vs_ai_revised__test"] = np.sort(np.concatenate([ai_side, human_side]))
    v1_split_col = df["extra"].map(lambda e: json.loads(e).get("v1_split", ""))
    splits["v1_test"] = idx(v1_split_col == "test")
    return splits
def split_frames(df, splits, condition):
    parts = []
    for side in ("train", "val", "test"):
        sel = splits.get(f"{condition}__{side}")
        if sel is None and side == "test":
            sel = splits.get(condition)
        parts.append(df.iloc[np.asarray(sel)].reset_index(drop=True) if sel is not None else df.iloc[0:0])
    return parts
if SPLITS_V2_JOBLIB.exists() and SPLITS_V2_JSON.exists() and not FORCE_REBUILD_V2 and not V2_REBUILT:
    splits_v2 = joblib.load(SPLITS_V2_JOBLIB)
    print(f"splits_v2 loaded: {len(splits_v2)} index arrays -> {SPLITS_V2_JOBLIB.name}")
else:
    if SPLITS_V2_JOBLIB.exists() and FROZEN_SPLITS is None:
        FROZEN_SPLITS = joblib.load(SPLITS_V2_JOBLIB)  # keep the frozen artifact for the parity check
    if FROZEN_DOC_IDS is None:
        FROZEN_DOC_IDS = df_v2["doc_id"].to_numpy()
    t0 = time.time()
    splits_v2 = build_splits_v2(df_v2, SEED)
    joblib.dump(splits_v2, SPLITS_V2_JOBLIB)
    summary = {
        "seed": SEED, "built_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "dataset_fingerprint": ds.dataset_fingerprint(df_v2),
        "counts": {name: int(len(sel)) for name, sel in sorted(splits_v2.items())},
        "assignment_counts": {s: int(len(splits_v2[f"in_domain_group__{s}"])) for s in ("train", "val", "test")},
        "class_mix_in_domain_test": df_v2.iloc[splits_v2["in_domain_group__test"]]["label"].value_counts().to_dict(),
        "assertions_ran": ["disjointness(in_domain_group)", "no_group_leak(in_domain_group)",
                           "no_unseen_generator_or_topic_in_train(cross_generator)",
                           "no_held_out_domain_in_train(cross_domain)",
                           "no_held_out_style_in_train(cross_style)"],
    }
    SPLITS_V2_JSON.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    V2_REBUILT = True
    print(f"splits_v2 rebuilt in {time.time() - t0:.0f}s: {len(splits_v2)} index arrays")
CONDITIONS = sorted({name.rsplit("__", 1)[0] for name in splits_v2})
print("conditions:", ", ".join(CONDITIONS))

### 2d — `data/style_features_v2.parquet`

Port of the original v2 feature builder: the ten served stylometric features
(coherence computed against a small TF-IDF space fitted on the corpus) plus the
Persian-specific v2 additions (ZWNJ / diacritic / digit / marker rates, hapax
ratio, word and sentence-length stats). POS-sequence and dependency-depth
features are absent on purpose — no offline Persian tagger/parser exists on
this host.

In [ ]:
EXTENDED_NAMES = ["zwnj_rate", "diacritic_rate", "digit_rate", "formal_marker_rate",
                  "informal_marker_rate", "informal_strong_marker_rate", "hapax_ratio",
                  "avg_word_len", "p90_sentence_words", "mean_sentence_words"]
if STYLE_FEATURES_V2_PARQUET.exists() and not FORCE_REBUILD_V2:
    style_v2 = pd.read_parquet(STYLE_FEATURES_V2_PARQUET)
    print(f"style_features_v2 loaded: {style_v2.shape}")
else:
    print(f"building style_features_v2 for {len(df_v2):,} documents ...")
    t0 = time.time()
    texts = df_v2["text"].tolist()
    coherence_vectorizer = TfidfVectorizer(max_features=30000, min_df=5, ngram_range=(1, 1))
    coherence_vectorizer.fit(texts)
    rows = []
    for i, text in enumerate(texts, start=1):
        rows.append({**compute_style_features(text, text, coherence_vectorizer),
                     **ds.extended_style_features(text)})
        if i % 2000 == 0:
            print(f"  {i:,}/{len(texts):,} ({time.time() - t0:.0f}s)")
    style_v2 = pd.DataFrame(rows, columns=[*STYLE_FEATURE_NAMES, *EXTENDED_NAMES])
    style_v2.insert(0, "doc_id", df_v2["doc_id"].to_numpy())
    style_v2.to_parquet(STYLE_FEATURES_V2_PARQUET, index=False)
    if DATASET_V2_MANIFEST.exists():  # manifest row, as the original builder did
        manifest = json.loads(DATASET_V2_MANIFEST.read_text(encoding="utf-8"))
        manifest["style_features_v2"] = {
            "file": STYLE_FEATURES_V2_PARQUET.name, "rows": int(len(style_v2)),
            "columns": list(style_v2.columns[1:]), "sha256": ds.sha256_file(STYLE_FEATURES_V2_PARQUET)}
        DATASET_V2_MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"wrote {STYLE_FEATURES_V2_PARQUET.name}: {style_v2.shape} ({time.time() - t0:.0f}s)")

### 2e — paraphrase / attack artifacts

`data/ai_revised.parquet` (machine-revised human documents = the `ai_revised`
class) and `data/paraphrase_attack.parquet` (the reserved AI test rows,
rewritten), both with per-row provenance (method, model, source doc id).

Ported here from the original paraphrase builder: the two fully offline rewriters
(`sentence_perturbation`, `punct_spacing_noise`) plus both drivers, compacted.
Not ported, because they need network/model downloads: `backtranslation`
(fa→en→fa via the Helsinki opus-mt checkpoints; capped at 60 sources/domain in
the frozen build), `lexical_substitution` (ParsBERT masked-LM), and
`llm_paraphrase` (five Persian prompt templates over an OpenAI-compatible
endpoint — never run for the frozen artifacts, no reachable host endpoint).
Deleting `ai_revised.parquet` triggers an offline-methods-only regeneration;
the attack set regenerates with `FORCE_REBUILD_ATTACK = True`. If you regenerate
revised rows, re-run cell 2b so the new rows enter `dataset_v2`.

In [ ]:
for path in (AI_REVISED_PARQUET, PARAPHRASE_ATTACK_PARQUET):
    if path.exists():
        frame = pd.read_parquet(path)
        print(f"{path.name}: {len(frame):,} rows; methods {frame['revision_type'].value_counts().to_dict()}")
    else:
        print(f"{path.name}: missing - the generation half of this cell can produce it")
_SENT_SPLIT = re.compile(r"(?<=[.!?؟…])\s+|\n+")
OFFLINE_METHODS = ("sentence_perturbation", "punct_spacing_noise")
def _split_sentences(text):
    return [s.strip() for s in _SENT_SPLIT.split(text) if s.strip()]
def _sentence_perturbation(text, rng, shuffle_rate=0.35):
    sentences = _split_sentences(text)
    if len(sentences) < 3:
        return text
    order = rng.permutation(len(sentences) - 1)
    moved = set(order[: max(1, int((len(sentences) - 1) * shuffle_rate))])
    reordered = [sentences[i + 1] for i in range(len(sentences) - 1) if i in moved] + \
        [sentences[i + 1] for i in range(len(sentences) - 1) if i not in moved]
    return " ".join([sentences[0], *reordered])
def _punct_spacing_noise(text, rng, rate=0.25):
    # typographic layer only: ZWNJ, komma spacing, digit form; no words change
    out = text
    if rng.random() < rate:
        out = out.replace(ds.ZWNJ, " ")
    if rng.random() < rate:
        out = out.replace(ds.ZWNJ, "")
    if rng.random() < rate:
        out = re.sub(r"\s*،\s*", "، ", out)
    if rng.random() < rate:
        out = out.translate({ord(str(d)): ord(str(d).translate(str.maketrans("0123456789", "۰۱۲۳۴۵۶۷۸۹")))
                             for d in range(10)})
    if rng.random() < rate:
        out = re.sub(r"\s+([.!?؟])", r"\1", out)
    return out
def _rewrite(text, method, rng):
    return (_sentence_perturbation(text, rng) if method == "sentence_perturbation"
            else _punct_spacing_noise(text, rng)), "rule-based"
def _paraphrase_record(source, text, method, model_id, attack=False):
    return {
        "doc_id": ds.stable_doc_id(text), "text": text,
        "label": ds.AI if attack else ds.AI_REVISED, "label_binary": 1,
        "domain": source["domain"], "source": ("attack:" if attack else "revision:") + method,
        "license": "derived-from-source", "origin": source["origin"], "generator": model_id,
        "provider": "local" if ("Helsinki" in model_id or "bert" in model_id.lower()) else "llm",
        "prompt_id": "", "prompt_style": source["prompt_style"], "topic_id": source["topic_id"],
        "revision_type": method, "paraphrase_method": method,
        "formality": ds.formality_label(text), "length_words": len(text.split()),
        "length_bucket": ds.length_bucket(len(text.split())), "author_group": source["author_group"],
        "non_native_flag": int(source["non_native_flag"]), "pii_scrubbed": int(source["pii_scrubbed"]),
        "split": "",
        "extra": json.dumps({"source_doc_id": source["doc_id"],
                             **({"original_label": source["label"]} if attack else {})}, ensure_ascii=False),
    }
def _generation_sources(candidates, min_words=8, max_words=300):
    # per-domain p10-p90 length window, clipped (a global window would drop the
    candidates = candidates.copy()
    candidates["_words"] = candidates["text"].str.split().str.len()
    quantiles = candidates.groupby("domain")["_words"].quantile([0.10, 0.90]).unstack()
    lo, hi = quantiles[0.10].clip(lower=min_words), quantiles[0.90].clip(upper=max_words)
    return candidates[candidates["_words"].between(candidates["domain"].map(lo), candidates["domain"].map(hi))]
def run_generation():
    frame = pd.read_parquet(DATASET_V2_PARQUET)
    rng = np.random.default_rng(SEED)
    if not AI_REVISED_PARQUET.exists():
        rows = []
        for domain, group in _generation_sources(frame[frame["label"] == ds.HUMAN]).groupby("domain"):
            if domain == "translated_subtitles":
                continue  # already translated; revising again would double-count
            for _, source in group.sample(min(200, len(group)), random_state=SEED).iterrows():
                for method in OFFLINE_METHODS:
                    text, model_id = _rewrite(source["text"], method, rng)
                    if text and text.strip() != source["text"].strip():
                        rows.append(_paraphrase_record(source, text, method, model_id))
        pd.DataFrame(rows).reset_index(drop=True).to_parquet(AI_REVISED_PARQUET, index=False)
        print(f"wrote revised rows ({', '.join(OFFLINE_METHODS)}) -> {AI_REVISED_PARQUET.name}")
    if FORCE_REBUILD_ATTACK or not PARAPHRASE_ATTACK_PARQUET.exists():
        attack_idx = joblib.load(SPLITS_V2_JOBLIB)["paraphrase_attention__test"]
        rows = []
        for _, source in frame.iloc[attack_idx].head(300).iterrows():
            for method in OFFLINE_METHODS:
                text, model_id = _rewrite(source["text"], method, rng)
                if text:
                    rows.append(_paraphrase_record(source, text, method, model_id, attack=True))
        pd.DataFrame(rows).reset_index(drop=True).to_parquet(PARAPHRASE_ATTACK_PARQUET, index=False)
        print(f"wrote attack rows ({', '.join(OFFLINE_METHODS)}) -> {PARAPHRASE_ATTACK_PARQUET.name}")
    # record what this notebook's offline regeneration covers
    manifest = json.loads(PARAPHRASE_MANIFEST.read_text(encoding="utf-8")) if PARAPHRASE_MANIFEST.exists() else {}
    manifest["notebook_offline_regeneration"] = {
        "methods": list(OFFLINE_METHODS),
        "note": "offline rule-based methods only; backtranslation / lexical_substitution / "
                "llm_paraphrase need network or an endpoint and were not re-run",
        "built_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    PARAPHRASE_MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
if FORCE_REBUILD_ATTACK or not (AI_REVISED_PARQUET.exists() and PARAPHRASE_ATTACK_PARQUET.exists()):
    run_generation()
else:
    print("paraphrase artifacts present - generation skipped (FORCE_REBUILD_ATTACK to regenerate)")

## 3 — Parity check

If a v2 rebuild ran in this session, the rebuilt splits must reproduce the
frozen `splits_v2.joblib` row-id-for-row-id per condition (doc ids are content
hashes, so the comparison is robust to row reordering); a mismatch is a **hard
error**. Without a rebuild, this cell still verifies the frozen artifacts: the
fingerprint in `splits_v2.json` must match the parquet, and an in-memory split
rebuild from the frozen parquet must reproduce every frozen condition (nothing
is written).

In [ ]:
def _ids_by_condition(doc_ids, splits):
    return {name: {doc_ids[i] for i in np.asarray(sel)} for name, sel in splits.items()}
def _split_mismatches(frozen_splits, frozen_ids, other_splits, other_ids):
    if set(frozen_splits) != set(other_splits):
        return [f"condition sets differ: {sorted(set(frozen_splits) ^ set(other_splits))}"]
    frozen_map, other_map = _ids_by_condition(frozen_ids, frozen_splits), _ids_by_condition(other_ids, other_splits)
    return [name for name in sorted(frozen_map) if frozen_map[name] != other_map[name]]
if V2_REBUILT and FROZEN_SPLITS is not None and FROZEN_DOC_IDS is not None:
    problems = _split_mismatches(FROZEN_SPLITS, FROZEN_DOC_IDS, splits_v2, df_v2["doc_id"].to_numpy())
    if problems:
        raise RuntimeError(
            f"rebuilt splits_v2 does not match the frozen artifact for {len(problems)} conditions: "
            f"{problems[:8]}. The rebuilt dataset or split logic drifted from the frozen build - "
            "inspect data/dataset_v2.manifest.json (dedup stats, dataset_fingerprint) before trusting it.")
    print(f"PARITY OK: rebuilt splits match the frozen artifact on all {len(FROZEN_SPLITS)} conditions")
else:
    problems = []
    summary = json.loads(SPLITS_V2_JSON.read_text(encoding="utf-8")) if SPLITS_V2_JSON.exists() else {}
    fingerprint = ds.dataset_fingerprint(df_v2)
    if summary.get("dataset_fingerprint") not in (None, fingerprint):
        problems.append(f"splits_v2.json fingerprint {summary['dataset_fingerprint'][:12]} != parquet {fingerprint[:12]}")
    problems += [f"in-memory rebuild differs for {name}" for name in _split_mismatches(
        splits_v2, df_v2["doc_id"].to_numpy(), build_splits_v2(df_v2, SEED), df_v2["doc_id"].to_numpy())]
    if problems:
        print("WARNING: frozen artifacts drifted:", *problems, sep="\n  ")
    else:
        print(f"frozen artifacts verified: fingerprint matches and an in-memory split rebuild "
              f"reproduces all {len(splits_v2)} conditions (nothing written)")

## 4 — Dataset overview

Counts per label / domain / origin and text-length statistics for `dataset_v2`,
plus the v1 balance and the available split conditions.

In [ ]:
overview = df_v2.groupby(["label", "domain", "origin"]).size().rename("rows").reset_index()
print(overview.sort_values(["label", "domain"]).to_string(index=False))
print()
length_stats = df_v2.groupby("label")["length_words"].agg(
    rows="size", mean="mean", median="median", p90=lambda s: s.quantile(0.9), max="max").round(1)
print(length_stats.to_string())
if PROCESSED_CSV.exists():
    v1_counts = pd.read_csv(PROCESSED_CSV, usecols=["aiGenerated"])["aiGenerated"].value_counts()
    print()
    print("v1 processedDataset.csv:", {("ai" if k == 1 else "human"): int(v) for k, v in v1_counts.items()})
print()
print("split conditions:", ", ".join(CONDITIONS))
print("splits_v2: dict[str, np.ndarray] of positional indices into dataset_v2.parquet;")
print("split_frames(df_v2, splits_v2, condition) -> (train, val, test) frames")